# BERT-Based News Categorization Engine
In this notebook, we develop an NLP pipeline to automatically classify news headlines. We leverage the AG News dataset to fine-tune a pre-trained **BERT** architecture, enabling it to accurately sort text into four distinct topics:
* Global News
* Sports Events
* Finance & Business
* Technology & Science

In [1]:
from datasets import load_dataset
from transformers import BertTokenizer
import torch

### 1. Data Acquisition: AG News Dataset
We fetch our training and testing data directly via the Hugging Face `datasets` library. The data consists of short news texts grouped by four main labels.

In [2]:
dataset = load_dataset("ag_news")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [3]:
dataset["train"][0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [4]:
dataset["train"].features

{'text': Value('string'),
 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}

### 2. Initializing the Tokenizer
**Text Tokenization Process**

Machine learning models cannot process raw text. We use the BERT tokenizer to translate our news headlines into numerical representations (input IDs and attention masks) that the transformer algorithm can understand.

In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Tokenizer Demonstration

In [6]:
example = dataset["train"][0]["text"]

tokens = tokenizer(example)

print(tokens)

{'input_ids': [101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813, 2395, 1005, 1055, 1040, 11101, 2989, 1032, 2316, 1997, 11087, 1011, 22330, 8713, 2015, 1010, 2024, 3773, 2665, 2153, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


### Applying Tokenization to the Full Dataset

In [7]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

### Inspecting the Processed Data

In [8]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

### Downsampling the Dataset for Faster Execution

In [9]:
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(300))
small_test = tokenized_datasets["test"].shuffle(seed=42).select(range(100))

### Dropping Redundant Features

In [10]:
small_train = small_train.remove_columns(["text"])
small_test = small_test.remove_columns(["text"])

### Configuring Tensor Format for PyTorch

In [11]:
small_train.set_format("torch")
small_test.set_format("torch")

### 3. Loading the Base BERT Architecture

Here, we import the foundational BERT model and configure its classification head to output exactly 4 categories.

In [12]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4,
    local_files_only=False
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 4. Setting Up Training Parameters

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1
)

### 5. Initializing the Model Trainer

In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
)

### 6. Executing Model Fine-Tuning
Using the Hugging Face Trainer API, we train the model on our prepared dataset. This step adjusts the BERT weights to specifically recognize news context and vocabulary.

In [15]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=0.7764981587727865, metrics={'train_runtime': 2094.2301, 'train_samples_per_second': 0.143, 'train_steps_per_second': 0.072, 'total_flos': 78934734028800.0, 'train_loss': 0.7764981587727865, 'epoch': 1.0})

### 7. Performance Evaluation
To gauge how well our model generalizes to unseen data, we calculate two key metrics:
* **Accuracy Score**
* **Weighted F1-Score**

In [16]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

predictions = trainer.predict(small_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

accuracy = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average="weighted")

preds

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


array([1, 2, 1, 2, 3, 2, 2, 2, 3, 2, 3, 1, 1, 0, 2, 3, 0, 1, 0, 3, 0, 2,
       1, 1, 0, 2, 2, 1, 2, 2, 2, 3, 3, 2, 3, 2, 1, 2, 2, 1, 1, 0, 0, 3,
       0, 1, 2, 1, 2, 1, 3, 0, 2, 2, 1, 2, 0, 3, 2, 1, 2, 1, 1, 2, 1, 3,
       2, 1, 1, 2, 0, 1, 0, 0, 0, 1, 1, 3, 3, 2, 0, 1, 1, 0, 1, 3, 1, 2,
       3, 2, 1, 1, 2, 1, 2, 3, 2, 3, 3, 3])

In [17]:
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.8200
F1 Score: 0.8175


### 8. Exporting the Trained Model

In [18]:
model.save_pretrained("news_bert_model")
tokenizer.save_pretrained("news_bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('news_bert_model/tokenizer_config.json', 'news_bert_model/tokenizer.json')

### 9. Interactive Web Application (Gradio)
To showcase the model's capability, we construct a user-friendly Gradio web app. This allows anyone to type a custom headline and instantly get an AI-driven topic prediction.

In [19]:
import torch
import gradio as gr

# Label Mapping
labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

# Build Gradio Function
def predict_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    predicted_class = torch.argmax(outputs.logits).item()
    return labels[predicted_class]

# Build Gradio Interface & Launch
interface = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=2, placeholder="Enter a news headline..."),
    outputs="text",
    title="News Topic Classifier (BERT)",
    description="Enter a news headline and the model will predict its category."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b4ce1cdb827e06dd9d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 10. Final Thoughts & Conclusion
This project successfully illustrates the power of transfer learning in NLP. By taking a foundational BERT model and fine-tuning it on the AG News dataset, we built a highly capable classifier for short text.

We handled text tokenization, model training, and rigorous evaluation using Accuracy and F1 metrics across four news domains. The addition of a Gradio interface bridges the gap between backend AI training and front-end user experience, proving the practical value of transformer models in automating text categorization tasks.